In [2]:
import os
import json
import warnings
import numpy as np
import xarray as xr
import proplot as pplt
warnings.filterwarnings('ignore')
pplt.rc.update({
    'savefig.dpi':900,
    'savefig.bbox':'tight',
    'savefig.pad_inches':0.02,
    'tick.minor':False,
    'font.size':9,
    'label.size':9,
    'tick.labelsize':9,
    'title.size':9,
    'abc.size':9,
    'legend.fontsize':9,
    'suptitle.size':9,
    'leftlabelsize':9,
    'toplabelsize':9,
    'leftlabel.weight':'normal',
    'toplabel.weight':'normal',
    'reso':'xx-hi'})

In [3]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
SPLITSDIR  = CONFIGS['filepaths']['splits']
WEIGHTSDIR = CONFIGS['filepaths']['weights']
PREDSDIR   = CONFIGS['filepaths']['predictions']
MODELS     = CONFIGS['experiments']
FIELDVARS  = MODELS['sr']['runs']['sr_gauss']['fieldvars']
NNSEEDS    = MODELS['nn']['seeds']
SPLIT      = 'test'
NBINS      = 40
MINSAMPLES = 30

EQPARAMS = {name:config['init'] for name,config in MODELS['sr']['optimizedeqs'].items()}
COLORS   = {name:config['color'] for name,config in MODELS['sr']['optimizedeqs'].items()}
COLORS.update({name:config['color'] for name,config in MODELS['pod']['runs'].items()})
LABELS   = {name:config['description'] for name,config in MODELS['sr']['optimizedeqs'].items()}
LABELS.update({name:config['description'] for name,config in MODELS['pod']['runs'].items()})

In [4]:
def kernel_integrate(fields,weights,dsig,mask=None):
    w = fields*weights[None,:,:]*dsig[None,None,:]
    if mask is not None: w = w*mask[:,None,:]
    return w.sum(axis=2)

def to_phys(norm):
    return np.expm1(tpstd*np.maximum(0.0,np.asarray(norm,dtype=float)))

def bin_1d(x,z,nbins=NBINS,minsamples=MINSAMPLES,plo=1,phi=99):
    finite = np.isfinite(x)&np.isfinite(z)
    x,z    = x[finite],z[finite]
    edges  = np.unique(np.percentile(x,np.linspace(plo,phi,nbins+1)))
    n      = len(edges)-1
    xi     = np.clip(np.digitize(x,edges)-1,0,n-1)
    counts = np.bincount(xi,minlength=n)
    sums   = np.bincount(xi,weights=z,minlength=n)
    return 0.5*(edges[:-1]+edges[1:]),np.where(counts>=minsamples,sums/counts,np.nan),counts

def bin_2d(x,y,z,nbins=NBINS,minsamples=MINSAMPLES,plo=1,phi=99):
    finite = np.isfinite(x)&np.isfinite(y)&np.isfinite(z)
    x,y,z  = x[finite],y[finite],z[finite]
    xedges = np.linspace(*np.percentile(x,[plo,phi]),nbins+1)
    yedges = np.linspace(*np.percentile(y,[plo,phi]),nbins+1)
    xi     = np.clip(np.digitize(x,xedges)-1,0,nbins-1)
    yi     = np.clip(np.digitize(y,yedges)-1,0,nbins-1)
    idx    = xi*nbins+yi
    counts = np.bincount(idx,minlength=nbins*nbins).reshape(nbins,nbins)
    sums   = np.bincount(idx,weights=z,minlength=nbins*nbins).reshape(nbins,nbins)
    return 0.5*(xedges[:-1]+xedges[1:]),0.5*(yedges[:-1]+yedges[1:]),np.where(counts>=minsamples,sums/counts,np.nan),counts

In [6]:
with open(os.path.join(SPLITSDIR,'stats.json'),'r',encoding='utf-8') as f:
    stats = json.load(f)
tpmean = float(stats['tp_mean'])
tpstd  = float(stats['tp_std'])

def flatten(da):
    if 'time' in da.dims: return da.transpose('time','lat','lon').values.ravel()
    return np.tile(da.values,(ntime,1,1)).ravel()

with xr.open_dataset(os.path.join(SPLITSDIR,f'norm_{SPLIT}.h5'),engine='h5netcdf') as ds:
    ntime,nlat,nlon = ds.sizes['time'],ds.sizes['lat'],ds.sizes['lon']
    nsig     = ds.sizes.get('sig',1)
    lat      = ds['lat'].values; lon = ds['lon'].values; dsig = ds['dsig'].values
    fields   = np.stack([ds[v].transpose('time','lat','lon','sig').values.reshape(-1,nsig)
                         for v in FIELDVARS],axis=1)
    surfmask = (ds['surfmask'].transpose('time','lat','lon','sig').values.reshape(-1,nsig)
                if 'surfmask' in ds else None)
    blnorm  = flatten(ds['bl'])
    lfnorm  = flatten(ds['lf'])  if 'lf'  in ds else None
    shfnorm = flatten(ds['shf']) if 'shf' in ds else None

kernels = []
for seed in NNSEEDS:
    wpath = os.path.join(WEIGHTSDIR,f'nn_gauss_{seed}_weights.nc')
    if os.path.exists(wpath):
        with xr.open_dataset(wpath,engine='h5netcdf') as ds:
            kernels.append(ds['k'].values)
ki = (np.mean([kernel_integrate(fields,k,dsig,surfmask) for k in kernels],axis=0)
      if kernels else fields.mean(axis=2))
rhk,thetaek,thetaestark = ki[:,0],ki[:,1],ki[:,2]

with xr.open_dataset(os.path.join(SPLITSDIR,f'{SPLIT}.h5'),engine='h5netcdf') as ds:
    obsflat  = ds.tp.transpose('time','lat','lon').values.ravel()
    lf2d     = ds['lf'].isel(time=0).values if 'lf' in ds else None
    landflat = np.tile(lf2d>=0.5,(ntime,1,1)).ravel() if lf2d is not None else None

def load_pred(name):
    path = os.path.join(PREDSDIR,f'{name}_{SPLIT}_predictions.nc')
    if not os.path.exists(path): return None
    with xr.open_dataset(path,engine='h5netcdf') as ds:
        da = ds['tp'].load()
    if 'seed' in da.dims: da = da.mean('seed')
    return da.transpose('time','lat','lon').values.ravel()

valid = np.isfinite(obsflat)&np.isfinite(rhk)&np.isfinite(thetaek)&np.isfinite(thetaestark)
print(f'{valid.sum():,} valid samples')

ValueError: Dimensions {'time'} do not exist. Expected one or more of ('lat', 'lon')

In [ ]:
c        = EQPARAMS['sr_lo']
rhcen,rhmeans,_ = bin_1d(rhk[valid],obsflat[valid],plo=0,phi=100)
rhsw      = np.linspace(rhcen[0],rhcen[-1],300)
srlocurve = to_phys(c['a']*np.exp(rhsw)+c['b'])

c         = EQPARAMS['sr_bl']
blcen,blmeans,_ = bin_1d(blnorm[valid],obsflat[valid],plo=0,phi=100)
blsw      = np.linspace(blcen[0],blcen[-1],300)
srblcurve = to_phys((blsw+c['a'])**3+c['b'])

podblpred = load_pred('pod_bl')
if podblpred is not None:
    _,podblmeans,_ = bin_1d(blnorm[valid],podblpred[valid],plo=0,phi=100)

In [ ]:
fig,axs = pplt.subplots(ncols=2,figwidth=4,sharex=False,sharey=True)
axs[0].scatter(rhcen,rhmeans,color='k',marker='.',s=20,zorder=0)
axs[0].plot(rhsw,srlocurve,color=COLORS['sr_lo'],lw=1.5,zorder=5,label=LABELS['sr_lo'])
axs[0].axhline(0,color='gray',lw=0.5)
axs[0].format(title='SR-LO',xlabel=r'$\widehat{\mathrm{RH}}$ (%)',xlim=(-5,105),xticks=25)
axs[0].legend(loc='ul',ncols=1)

finite = np.isfinite(blmeans)
axs[1].scatter(blcen[finite],blmeans[finite],color='k',marker='.',s=20,zorder=0)
axs[1].plot(blsw,srblcurve,color=COLORS['sr_bl'],lw=1.5,zorder=5,label=LABELS['sr_bl'])
if podblpred is not None:
    axs[1].plot(blcen,podblmeans,color=COLORS['pod_bl'],lw=1.5,zorder=4,label=LABELS['pod_bl'])
axs[1].axhline(0,color='gray',lw=0.5)
axs[1].format(title='SR-BL',xlabel=r'$\mathit{B_L}$ (m s$^{-2}$)',xlim=(-0.55,0.05),xticks=0.25)
axs[1].legend(loc='ul',ncols=1)
axs.format(abc=True,grid=False,ylabel='Total Precipitation (mm)',
           ylim=(-1,16),yticks=5,titleloc='l')
pplt.show()
fig.save('../figs/fig_3.jpg')

In [ ]:
c  = EQPARAMS['sr_med']
M  = rhk[valid]
I  = thetaek[valid]-c['b']*thetaestark[valid]-c['c']
P  = obsflat[valid]
moist = M>=I; insta = ~moist
print(f'Moisture-dominated (M≥I): {100*moist.mean():.1f}%  mean P={P[moist].mean():.3f} mm')
print(f'Instability-dominated (I>M): {100*insta.mean():.1f}%  mean P={P[insta].mean():.3f} mm')
if landflat is not None:
    lv = landflat[valid].astype(bool)
    for lbl,lm in [('Land',lv),('Ocean',~lv)]:
        for rlbl,rm in [('Moisture',moist),('Instability',insta)]:
            mm = lm&rm
            print(f'  {lbl} {rlbl}: {100*mm.sum()/lm.sum():.1f}%  mean P={P[mm].mean():.3f}')

In [ ]:
c   = EQPARAMS['sr_med']
M   = rhk[valid]; I = thetaek[valid]-c['b']*thetaestark[valid]-c['c']; P = obsflat[valid]
Mlo,Mhi = float(np.percentile(M,1)),float(np.percentile(M,99))
Ilo,Ihi = float(np.percentile(I,1)),float(np.percentile(I,99))
axlo = min(Mlo,Ilo); axhi = max(Mhi,Ihi)
xc,yc,obsbin,_ = bin_2d(M,I,P,nbins=35)
xc,yc,srbin,_  = bin_2d(M,I,to_phys(c['a']*np.maximum(M,I)**3),nbins=35)
density = np.log10(np.histogram2d(M,I,bins=[np.linspace(Mlo,Mhi,36),np.linspace(Ilo,Ihi,36)])[0].T.clip(1))
NG = 200
Mg,Ig  = np.meshgrid(np.linspace(Mlo,Mhi,NG),np.linspace(Ilo,Ihi,NG))
Pgrid  = to_phys(c['a']*np.maximum(Mg,Ig)**3)
Mfull  = rhk.reshape(ntime,nlat,nlon)
Ifull  = (thetaek-c['b']*thetaestark-c['c']).reshape(ntime,nlat,nlon)
frac2d = (Mfull>=Ifull).mean(axis=0)
latlim = (float(lat.min()),float(lat.max()))
lonlim = (float(lon.min()),float(lon.max()))
xlabelM = r'$\mathit{M}$ (Standardized Anomaly)'
ylabelI = r'$\mathit{I}$ (Standardized Anomaly)'
kw    = dict(cmap='ColdHot_r',cmap_kw={'left':0.5},vmin=0,vmax=4,levels=18,extend='max')
kwmap = dict(coast=True,lonlim=lonlim,lonlines=[65,75,85],lonlabels='b',
             latlim=latlim,latlines=[10,15,20],latlabels='l',grid=False)
fig,axs = pplt.subplots([[1,2],[3,4]],figwidth=5.5,proj={4:'cyl'},share=False,hspace=8)
m0 = axs[0].pcolormesh(np.linspace(Mlo,Mhi,NG),np.linspace(Ilo,Ihi,NG),Pgrid,**kw)
axs[0].plot([axlo,axhi],[axlo,axhi],'k--',lw=1)
axs[0].text(Mhi-0.13*(Mhi-Mlo),Ilo+0.08*(Ihi-Ilo),'moisture-dominated',ha='right',va='bottom')
axs[0].text(Mlo+0.05*(Mhi-Mlo),Ihi-0.12*(Ihi-Ilo),'instability-dominated',ha='left',va='top')
axs[0].format(xlabel=xlabelM,ylabel=ylabelI,title='SR-MED',xlim=(Mlo,Mhi),ylim=(Ilo,Ihi))
axs[1].pcolormesh(xc,yc,obsbin.T,**kw)
axs[1].plot([axlo,axhi],[axlo,axhi],'k--',lw=1)
axs[1].format(xlabel=xlabelM,ylabel='',yticklabels=[],title='ERA5',
              xlim=(Mlo,Mhi),ylim=(Ilo,Ihi),facecolor='gray1')
m2 = axs[2].pcolormesh(xc,yc,density,cmap='Grays',vmin=0)
axs[2].plot([axlo,axhi],[axlo,axhi],'k--',lw=0.8)
axs[2].format(xlabel=xlabelM,ylabel=ylabelI,title='Sample Density',xlim=(Mlo,Mhi),ylim=(Ilo,Ihi))
axs[2].colorbar(m2,loc='b',label=r'log$_{10}$($\mathit{N}$)')
m3 = axs[3].pcolormesh(lon,lat,frac2d,cmap='ColdHot_r',vmin=0,vmax=1)
axs[3].format(title='Moisture-Dominated Fraction',**kwmap)
axs[3].colorbar(m3,loc='b',ticks=0.2,label=r'Fraction ($\mathit{M}$ ≥ $\mathit{I}$)')
fig.format(abc=True,titleloc='l')
fig.canvas.draw()
figh  = fig.get_figheight()
left  = min(axs[0].get_position().x0,axs[1].get_position().x0)
right = max(axs[0].get_position().x1,axs[1].get_position().x1)
btop  = min(axs[0].get_position().y0,axs[1].get_position().y0)
tbot  = max(axs[2].get_position().y1,axs[3].get_position().y1)
cax   = fig.add_axes([left,tbot+0.5*(btop-tbot)-0.09/figh,right-left,0.18/figh])
cb    = fig.colorbar(m0,cax=cax,orientation='horizontal',
                     label='Total Precipitation (mm)',extend='max')
cb.ax.tick_params(labelsize=9)
pplt.show()
fig.save('../figs/fig_4.jpg')

In [ ]:
c    = EQPARAMS['sr_hi']
Mhi  = rhk[valid]; Ihi = thetaek[valid]+c['b']*thetaestark[valid]+c['c']
T    = np.maximum(Mhi,Ihi)
S    = np.maximum(lfnorm[valid],shfnorm[valid])
P    = obsflat[valid]
Tlo,Thi = float(np.percentile(T,1)),float(np.percentile(T,99))
Slo,Shi = float(np.percentile(S,1)),float(np.percentile(S,99))
lfmean2d  = lfnorm.reshape(ntime,nlat,nlon).mean(axis=0)
shfmean2d = shfnorm.reshape(ntime,nlat,nlon).mean(axis=0)
maxmean2d = np.maximum(lfnorm,shfnorm).reshape(ntime,nlat,nlon).mean(axis=0)
climlim   = float(np.nanmax(np.abs([lfmean2d,shfmean2d,maxmean2d])))
supp      = to_phys((c['a']*T)**3)-to_phys((c['a']*T+c['d']*S)**3)
Tcen,suppmeans,_ = bin_1d(T,supp)

In [ ]:
c      = EQPARAMS['sr_hi']
latlim = (float(lat.min()),float(lat.max()))
lonlim = (float(lon.min()),float(lon.max()))
Ssw    = np.linspace(Slo,Shi,200)
Tpcts  = [25,50,75,90]
Tfixed = np.percentile(T,Tpcts)
kwmap  = dict(coast=True,lonlim=lonlim,latlim=latlim,
              latlines=[10,15,20],lonlines=[65,75,85],grid=False)
kwsfc  = dict(cmap='ColdHot',vmin=-climlim,vmax=climlim)
fig,axs = pplt.subplots([[1,1,2,2,3,3],[0,4,4,5,5,0]],figwidth=6.5,
                         proj={1:'cyl',2:'cyl',3:'cyl'},share=False,wspace=0.75,hspace=9)
ms = axs[0].pcolormesh(lon,lat,lfmean2d,**kwsfc)
axs[1].pcolormesh(lon,lat,shfmean2d,**kwsfc)
axs[2].pcolormesh(lon,lat,maxmean2d,**kwsfc)
axs[0].format(title=r'$\widetilde{\mathrm{LF}}$',latlabels='l',lonlabels='b',**kwmap)
axs[1].format(title=r'$\widetilde{\mathrm{SHF}}$',lonlabels='b',**kwmap)
axs[2].format(title='$\mathit{S}$',lonlabels='b',**kwmap)
axs[3].scatter(Tcen,suppmeans,color=COLORS['sr_hi'],s=20,zorder=4)
axs[3].axhline(0,color='gray',lw=0.5)
axs[3].format(xlabel='$\mathit{T}$',xlim=(-1.75,1.25),xticks=0.5,
              ylabel='Total Precipitation (mm)',ylim=(-0.1,3.1),yticks=1,
              title='Suppression by $\mathit{\lambda S}$')
for perc,tv,ls in zip(Tpcts,Tfixed,['-','--',':','-.']):
    axs[4].plot(Ssw,to_phys((c['a']*tv+c['d']*Ssw)**3),color=COLORS['sr_hi'],ls=ls,
                label=fr'$\mathit{{T}}_{{{perc}}}$',lw=1)
axs[4].axhline(0,color='gray',lw=0.5)
axs[4].sharey(axs[3])
axs[4].format(xlabel='$\mathit{\lambda S}$',xlim=(-0.5,1.75),xticks=0.4,
              title='SR-HI at Fixed $\mathit{T}$')
axs[4].tick_params(axis='y',labelleft=False)
axs[4].legend(loc='ur',ncols=2,fontsize=7)
fig.format(abc=True,titleloc='l',grid=False)
fig.canvas.draw()
figh  = fig.get_figheight()
left  = axs[0].get_position().x0
right = axs[2].get_position().x1
btop  = min(axs[0].get_position().y0,axs[1].get_position().y0,axs[2].get_position().y0)
tbot  = max(axs[3].get_position().y1,axs[4].get_position().y1)
cax   = fig.add_axes([left,tbot+0.6*(btop-tbot)-0.09/figh,right-left,0.18/figh])
cb    = fig.colorbar(ms,cax=cax,orientation='horizontal',
                     label='Standardized Anomaly',extend='both')
cb.ax.tick_params(labelsize=9)
pplt.show()
fig.save('../figs/fig_5.jpg')